# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata
print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each entity (record set, field, column, etc.) is referenced by its `@id`.

Let's enumerate the available RecordSets and fields in the dataset.

In [ ]:
# Display RecordSets and their fields
record_sets = dataset.metadata.record_set

print("Available RecordSets (@id and name):")
for rs in record_sets:
    print(f"  - @id: {rs['@id']} | name: {rs.get('name', '<no name>')}")

print("\nFields per RecordSet:")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    if 'field' in rs:
        for f in rs['field']:
            print(f"  - field @id: {f['@id']} | name: {f.get('name', '<no name>')}")
    else:
        print("  (No fields listed)")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract all available record sets (`@id`) into pandas DataFrames, referenced by their `@id`.

In [ ]:
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_set]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found for RecordSet @id: {record_set_id}")
        continue
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"DataFrame for RecordSet @id: {record_set_id} loaded. Columns:")
    print(dataframes[record_set_id].columns.tolist())
    print(dataframes[record_set_id].head(), "\n")

# As an example, let's pick the first RecordSet to continue with EDA and visualization
if len(dataframes) > 0:
    example_record_set_id = list(dataframes.keys())[0]
    df = dataframes[example_record_set_id]
    print(f"Selected RecordSet @id for analysis: {example_record_set_id}")
else:
    df = None
    example_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Here we demonstrate filtering, normalization, and grouping for fields referenced by their `@id`.

In [ ]:
# Check for numeric fields (prefer GAD-7 or PHQ-9 scores)
if df is not None:
    numeric_field_candidates = []
    # Find fields likely to be numeric by checking field names or sample values
    for col in df.columns:
        # Try to infer numeric fields by column name or type
        if 'score' in col.lower() or 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower():
            numeric_field_candidates.append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_candidates.append(col)

    if len(numeric_field_candidates) == 0:
        print("No obvious numeric fields found.\n")
    else:
        print("Numeric field candidates:", numeric_field_candidates)
    
    # Choose the first numeric field for demonstration
    numeric_field = numeric_field_candidates[0] if len(numeric_field_candidates) > 0 else df.columns[0]

    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Choose a group field (prefer demographic categorical)
    group_field_candidates = [col for col in df.columns if 'age' in col.lower() or 'gender' in col.lower() or 'education' in col.lower() or pd.api.types.is_object_dtype(df[col])]
    group_field = group_field_candidates[0] if len(group_field_candidates) > 0 else None

    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of the selected numeric field and its relationship with a group field using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides survey records on mental health indicators from Kilifi County.
- We loaded and inspected available record sets and fields using their `@id` via the Croissant schema.
- Numeric and demographic fields were analyzed and visualized, demonstrating possible approaches to EDA.
- The data may contain minimal missing entries (as per metadata notes) and shows opportunities for research on mental health trends in Kenya.